# XLM-RoBERTa Fine-tuning

本 notebook 用於在 Google Colab GPU 上 fine-tune `xlm-roberta-base`，訓練中英文標題黨二元分類模型。

流程會從 Google Drive 掛載資料、載入 `dataset/processed/` 的 train/valid/test CSV，fine-tune 模型後評估並將 checkpoint 存回 Drive。

## 環境安裝

In [ ]:
# 在 Colab 上安裝所需套件；本機執行時可略過。
!pip install -q transformers datasets accelerate scikit-learn deep-translator

## 掛載 Google Drive

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

## 路徑與超參數設定

In [ ]:
from pathlib import Path

DRIVE_PROJECT    = Path("/content/drive/MyDrive/NLP_Project")
PROCESSED_DIR    = DRIVE_PROJECT / "dataset" / "processed"
MODEL_OUTPUT_DIR = DRIVE_PROJECT / "models" / "xlm-roberta-clickbait"

MODEL_NAME    = "xlm-roberta-base"
MAX_LENGTH    = 256
BATCH_SIZE    = 16
NUM_EPOCHS    = 3
LEARNING_RATE = 2e-5
SEED          = 42

TOKEN_DROPOUT_PROB = 0.1  # 訓練時隨機將部分 token 換成 <mask>，提升泛化 （0.0/0.1）
CONTENT_CROP_PROB  = 0.5  # 訓練時有此機率對 content 從隨機位置截斷（0.0/0.5）
CLICKBAIT_WEIGHT   = 1.15  # clickbait(label=1) 的 loss 權重；>1 讓模型更敢判 clickbait（降 FN、升 FP）。1.0=關閉
TW_NEG_KEEP_RATIO  = 1.0   # tw_neg_only(_twneg, 全 label=0) 保留比例：1.0=全留 / 0.5 / 0.25 / 0.0=全刪。Bug3 已馴服可降低以降 FN
BACKTRANSLATE      = False  # 改成 True 才會執行回譯增強；預設關閉
BT_SAMPLE_RATIO    = 0.3    # 只對訓練集的 30% 做回譯，控制 API 呼叫量
THRESHOLD = 0.50

print("Processed dir:", PROCESSED_DIR)
print("Model output:", MODEL_OUTPUT_DIR)

## G5 / G7 / G8 訓練開關

- `USE_ADVERSARIAL_TONE = True, USE_ADVERSARIAL_G8 = False` → G7（G5 + tone 對抗資料）
- `USE_ADVERSARIAL_G8 = True` → G8（G7 + tw_pair/cb 對抗資料，覆蓋 G7 設定）
- 兩者皆 `False` → G5（純原始訓練集）

In [ ]:
USE_ADVERSARIAL_TONE = True   # G7 開關；False 時等同 G5
USE_ADVERSARIAL_G8   = False  # G8 開關；True 時為 G8（會覆蓋 G7 設定）

if USE_ADVERSARIAL_G8:
    RUN_TAG = "g8"
elif USE_ADVERSARIAL_TONE:
    RUN_TAG = "g7"
else:
    RUN_TAG = "g5"

MODEL_OUTPUT_DIR = DRIVE_PROJECT / f"models/xlm-roberta-clickbait-{RUN_TAG}"
METRICS_OUT = DRIVE_PROJECT / f"results/{RUN_TAG}_transformer_metrics.json"

print(f"Run tag: {RUN_TAG}")
print(f"Model output: {MODEL_OUTPUT_DIR}")
print(f"Metrics output: {METRICS_OUT}")

## 匯入套件

In [ ]:
import random

import numpy as np
import pandas as pd
import torch
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
)
from torch.utils.data import DataLoader, Dataset
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    get_linear_schedule_with_warmup,
)


def set_seed(seed):
    """固定所有隨機種子，讓訓練結果可重現"""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


set_seed(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)

## 載入資料

## 資料增強：回譯（Back-translation）

此 cell 為**選擇性執行**，用 Google Translate API 對訓練集做回譯增強：
- 中文標題：中文 → 英文 → 中文，產生語意相同但用詞不同的新樣本
- 英文標題：英文 → 中文 → 英文，同理

回譯結果存成 `train_augmented.csv`，之後載入資料時可選擇使用。
若不想增強或 API 額度不足，直接跳過此 cell 即可。

In [ ]:
!pip install -q deep-translator

In [ ]:
# 【選擇性執行】回譯只對 title 做；content 太長，翻譯成本高且標題才是分類的關鍵特徵
BT_OUTPUT_PATH = PROCESSED_DIR / "train_augmented.csv"


def back_translate_title(title, src_lang):
    """將 title 翻譯成對方語言再翻回來，產生語意相同但用詞不同的新標題。"""
    from deep_translator import GoogleTranslator
    pivot = "en" if src_lang == "zh" else "zh"
    try:
        translated = GoogleTranslator(source=src_lang, target=pivot).translate(title)
        back       = GoogleTranslator(source=pivot,    target=src_lang).translate(translated)
        return back
    except Exception:
        # API 失敗時保留原文，不讓整批增強中斷
        return title


if BACKTRANSLATE:
    import time
    from deep_translator import GoogleTranslator  # noqa: F401 — 提前 import 驗證安裝是否成功

    _train_df = pd.read_csv(PROCESSED_DIR / "train.csv")
    sample = _train_df.sample(frac=BT_SAMPLE_RATIO, random_state=SEED)
    bt_rows = []

    for i, (_, row) in enumerate(sample.iterrows()):
        new_title = back_translate_title(row["title"], row["lang"])
        bt_rows.append({
            "id":      row["id"] + "_bt",
            "lang":    row["lang"],
            "title":   new_title,
            "content": row["content"],
            "label":   row["label"],
            "source":  row["source"],
        })
        if (i + 1) % 50 == 0:
            print(f"  Back-translated {i+1}/{len(sample)}")
            time.sleep(1)  # 避免觸發 API rate limit

    bt_df = pd.DataFrame(bt_rows)
    augmented = pd.concat([_train_df, bt_df], ignore_index=True).sample(frac=1, random_state=SEED)
    augmented.to_csv(BT_OUTPUT_PATH, index=False, encoding="utf-8-sig")
    print(f"Augmented train saved: {len(augmented):,} rows -> {BT_OUTPUT_PATH}")
else:
    print("Back-translation skipped (BACKTRANSLATE=False).")

In [ ]:
# 若有做回譯增強且檔案存在，就用增強後的訓練集；否則用原始 train.csv
_aug_path = PROCESSED_DIR / "train_augmented.csv"
train_path = _aug_path if BACKTRANSLATE else PROCESSED_DIR / "train.csv"

train_df = pd.read_csv(train_path)
valid_df = pd.read_csv(PROCESSED_DIR / "valid.csv")
test_df  = pd.read_csv(PROCESSED_DIR / "test.csv")

print(f"Train source: {train_path.name}")

# G8：併入 G8 對抗資料（包含 tone + tw_pair + cb）
if USE_ADVERSARIAL_G8:
    adv_path = PROCESSED_DIR / "adversarial_g8.csv"
    adv = pd.read_csv(adv_path)
    train_df = pd.concat([train_df, adv], ignore_index=True)
    print(f"G8: 併入對抗資料 {len(adv)} 筆，總訓練筆數 {len(train_df)}")
# G7：併入 tone 對抗資料
elif USE_ADVERSARIAL_TONE:
    adv_path = PROCESSED_DIR / "adversarial_tone.csv"
    adv = pd.read_csv(adv_path)
    train_df = pd.concat([train_df, adv], ignore_index=True)
    print(f"G7: 併入對抗資料 {len(adv)} 筆，總訓練筆數 {len(train_df)}")

# 確認欄位完整、label 只有 0/1、中英文都有進來
for name, df in [("train", train_df), ("valid", valid_df), ("test", test_df)]:
    assert set(df.columns) >= {"id", "lang", "title", "content", "label"}, f"{name} missing columns"
    assert set(df["label"].unique()).issubset({0, 1}), f"{name} has unexpected labels"
    assert {"zh", "en"}.issubset(set(df["lang"].unique())), f"{name} missing a language"
    print(f"{name}: {len(df):,} rows | label 0={df['label'].eq(0).sum():,} label 1={df['label'].eq(1).sum():,}")

In [ ]:
# ── 【可選】刪減 tw_neg_only，測 Bug3 已馴服後能否再降 FN ──
# 放在「載入資料」cell 之後執行。改 KEEP_RATIO 一個數即可切 A/B/C/D。
# 1.0=全留(A) / 0.5=B / 0.25=C / 0.0=全刪(D)。不動任何 csv。
TW_NEG_KEEP_RATIO = 0.5

if USE_ADVERSARIAL_G8 and TW_NEG_KEEP_RATIO < 1.0:
    is_twneg = train_df["id"].astype(str).str.endswith("_twneg")
    twneg = train_df[is_twneg]
    keep_n = int(round(len(twneg) * TW_NEG_KEEP_RATIO))
    twneg_keep = twneg.sample(n=keep_n, random_state=SEED) if keep_n > 0 else twneg.iloc[0:0]
    train_df = pd.concat([train_df[~is_twneg], twneg_keep], ignore_index=True)
    print(f"tw_neg_only: {int(is_twneg.sum())} → 保留 {keep_n} (ratio={TW_NEG_KEEP_RATIO})")
    print(f"訓練筆數: {len(train_df)} | label 0={train_df['label'].eq(0).sum()} label 1={train_df['label'].eq(1).sum()}")
else:
    print(f"tw_neg_only 未刪減 (G8={USE_ADVERSARIAL_G8}, ratio={TW_NEG_KEEP_RATIO})")


## Dataset、DataLoader 與資料增強

訓練集使用兩種 on-the-fly 增強：
- **Token Dropout**：隨機將部分 token 換成 `<mask>`，讓模型更不依賴單一詞彙
- **Content 隨機截斷**：每次 epoch content 從不同位置截斷，讓模型看到內文的不同片段

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
MASK_TOKEN_ID = tokenizer.mask_token_id


def augment_content(content, crop_prob):
    """有 crop_prob 的機率從隨機位置截取 content，讓模型見到內文不同片段"""
    if not content or random.random() > crop_prob:
        return content
    words = content.split()
    if len(words) <= 1:
        return content
    start = random.randint(0, len(words) // 2)
    return " ".join(words[start:])


def apply_token_dropout(input_ids, attention_mask, mask_id, dropout_prob):
    """把非特殊 token 以 dropout_prob 的機率換成 <mask>，增強模型泛化"""
    # attention_mask=0 的位置是 padding，不做替換
    special_ids = {tokenizer.cls_token_id, tokenizer.sep_token_id,
                   tokenizer.pad_token_id, tokenizer.eos_token_id}
    mask = (
        (torch.rand_like(input_ids.float()) < dropout_prob)
        & attention_mask.bool()
        & ~torch.isin(input_ids, torch.tensor(list(special_ids)))
    )
    input_ids[mask] = mask_id
    return input_ids


class ClickbaitDataset(Dataset):
    """把 DataFrame 包成 PyTorch Dataset，輸入格式為 title </s> content"""

    def __init__(self, df, tokenizer, max_length, augment=False):
        self.labels   = df["label"].tolist()
        self.titles   = df["title"].fillna("").tolist()
        self.contents = df["content"].fillna("").tolist()
        self.tokenizer  = tokenizer
        self.max_length = max_length
        # augment=True 只在訓練集使用；valid/test 不做增強，確保評估穩定
        self.augment = augment

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        title   = self.titles[idx]
        content = self.contents[idx]

        if self.augment:
            content = augment_content(content, CONTENT_CROP_PROB)

        enc = self.tokenizer(
            title,
            content,
            max_length=self.max_length,
            padding="max_length",
            truncation=True,
            return_tensors="pt",
        )
        input_ids      = enc["input_ids"].squeeze(0)
        attention_mask = enc["attention_mask"].squeeze(0)

        if self.augment:
            input_ids = apply_token_dropout(
                input_ids.clone(), attention_mask, MASK_TOKEN_ID, TOKEN_DROPOUT_PROB
            )

        return {
            "input_ids":      input_ids,
            "attention_mask": attention_mask,
            "labels":         torch.tensor(self.labels[idx], dtype=torch.long),
        }


# 訓練集開啟增強；valid/test 關閉，確保評估可重現
train_dataset = ClickbaitDataset(train_df, tokenizer, MAX_LENGTH, augment=True)
valid_dataset = ClickbaitDataset(valid_df, tokenizer, MAX_LENGTH, augment=False)
test_dataset  = ClickbaitDataset(test_df,  tokenizer, MAX_LENGTH, augment=False)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=2, pin_memory=True)
valid_loader = DataLoader(valid_dataset, batch_size=BATCH_SIZE,
                          num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE,
                          num_workers=2, pin_memory=True)

print(f"train batches: {len(train_loader)} | valid batches: {len(valid_loader)} | test batches: {len(test_loader)}")

## 基本測試

In [ ]:
# 取一個 batch 確認 shape 與 label 值域正常
sample_batch = next(iter(train_loader))
assert sample_batch["input_ids"].shape == (BATCH_SIZE, MAX_LENGTH)
assert sample_batch["attention_mask"].shape == (BATCH_SIZE, MAX_LENGTH)
assert sample_batch["labels"].max().item() <= 1
assert sample_batch["labels"].min().item() >= 0
print("Dataset smoke test passed.")
print("input_ids shape:", sample_batch["input_ids"].shape)

## 載入模型

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2,
)
model = model.to(DEVICE)

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total params: {total_params:,} | Trainable: {trainable_params:,}")

## 訓練

In [ ]:
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE)

total_steps  = len(train_loader) * NUM_EPOCHS
warmup_steps = int(total_steps * 0.1)  # 前 10% steps 做 warmup

# LR 從 0 線性爬升到 LEARNING_RATE（warmup），再線性衰減到 0（剩餘訓練）
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_steps,
)

# class weight：[non-clickbait, clickbait]。CLICKBAIT_WEIGHT>1 提高 label=1 的 loss 權重
# 用自訂 criterion 取代 HF 內建無權重的 loss
class_weights = torch.tensor([1.0, CLICKBAIT_WEIGHT], dtype=torch.float, device=DEVICE)
criterion = torch.nn.CrossEntropyLoss(weight=class_weights)

print(f"Total steps: {total_steps} | Warmup steps: {warmup_steps}")
print(f"Class weights [non, clickbait]: {class_weights.tolist()}")

In [ ]:
def evaluate(model, loader, device):
    """在給定 DataLoader 上跑推理，回傳 loss、accuracy 與 macro F1。
    loss 用與訓練相同的加權 criterion，讓 train/val loss 可比。"""
    model.eval()
    all_preds, all_labels, total_loss = [], [], 0.0

    with torch.no_grad():
        for batch in loader:
            input_ids      = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels         = batch["labels"].to(device)

            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            loss = criterion(outputs.logits, labels)
            total_loss += loss.item()

            preds = outputs.logits.argmax(dim=-1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    avg_loss = total_loss / len(loader)
    acc = accuracy_score(all_labels, all_preds)
    f1  = f1_score(all_labels, all_preds, average="macro")
    return avg_loss, acc, f1


best_valid_f1 = 0.0
history = {"train_loss": [], "val_loss": [], "val_acc": [], "val_f1": []}

for epoch in range(1, NUM_EPOCHS + 1):
    model.train()
    train_loss = 0.0

    for step, batch in enumerate(train_loader, 1):
        input_ids      = batch["input_ids"].to(DEVICE)
        attention_mask = batch["attention_mask"].to(DEVICE)
        labels         = batch["labels"].to(DEVICE)

        optimizer.zero_grad()
        # 不傳 labels 給 model，改用加權 criterion 自行計算 loss
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        loss = criterion(outputs.logits, labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)  # 防止梯度爆炸
        optimizer.step()
        scheduler.step()  # 每個 step 更新 LR

        train_loss += loss.item()
        if step % 500 == 0:
            current_lr = optimizer.param_groups[0]["lr"]
            print(f"  Epoch {epoch} step {step}/{len(train_loader)} | loss={train_loss/step:.4f} | lr={current_lr:.2e}")

    avg_train_loss = train_loss / len(train_loader)
    val_loss, val_acc, val_f1 = evaluate(model, valid_loader, DEVICE)

    history["train_loss"].append(avg_train_loss)
    history["val_loss"].append(val_loss)
    history["val_acc"].append(val_acc)
    history["val_f1"].append(val_f1)

    current_lr = optimizer.param_groups[0]["lr"]
    print(f"Epoch {epoch} | train_loss={avg_train_loss:.4f} | val_loss={val_loss:.4f} | val_acc={val_acc:.4f} | val_f1={val_f1:.4f} | lr={current_lr:.2e}")

    # 只存 validation F1 最好的 checkpoint
    if val_f1 > best_valid_f1:
        best_valid_f1 = val_f1
        model.save_pretrained(MODEL_OUTPUT_DIR)
        tokenizer.save_pretrained(MODEL_OUTPUT_DIR)
        print(f"  -> Best model saved (val_f1={val_f1:.4f})")

print(f"\nTraining done. Best val F1: {best_valid_f1:.4f}")

In [ ]:
import matplotlib.pyplot as plt

# 檢查 results 資料夾是否存在
RESULTS_DIR = DRIVE_PROJECT / "results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

epochs_range = range(1, NUM_EPOCHS + 1)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

# Loss curve
ax1.plot(epochs_range, history["train_loss"], marker="o", label="Train Loss")
ax1.plot(epochs_range, history["val_loss"],   marker="o", label="Val Loss")
ax1.set_xlabel("Epoch"); ax1.set_ylabel("Loss")
ax1.set_title("Train / Val Loss"); ax1.legend(); ax1.grid(True)

# F1 curve
ax2.plot(epochs_range, history["val_f1"],  marker="o", label="Val Macro F1", color="green")
ax2.plot(epochs_range, history["val_acc"], marker="o", label="Val Accuracy",  color="orange")
ax2.set_xlabel("Epoch"); ax2.set_ylabel("Score")
ax2.set_title("Val F1 & Accuracy"); ax2.legend(); ax2.grid(True)

fig.tight_layout()
curve_path = DRIVE_PROJECT / "results" / "transformer_training_curve.png"
fig.savefig(curve_path, dpi=150)
plt.show()
print("Training curve saved to:", curve_path)

## 測試集評估

In [ ]:
# 從 Drive 重新載入最佳 checkpoint 來做最終評估，確保評估的是存下來的模型
best_model = AutoModelForSequenceClassification.from_pretrained(MODEL_OUTPUT_DIR)
best_model = best_model.to(DEVICE)

best_model.eval()
all_preds, all_labels = [], []

with torch.no_grad():
    for batch in test_loader:
        input_ids      = batch["input_ids"].to(DEVICE)
        attention_mask = batch["attention_mask"].to(DEVICE)
        labels         = batch["labels"].to(DEVICE)

        outputs = best_model(input_ids=input_ids, attention_mask=attention_mask)
        probs = torch.softmax(outputs.logits, dim=-1)[:, 1]   # p(clickbait)
        preds = (probs >= THRESHOLD).long()
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

print("=== Test Set Results ===")
print(classification_report(all_labels, all_preds, target_names=["non-clickbait", "clickbait"]))
print("Confusion matrix:")
print(confusion_matrix(all_labels, all_preds))

## 儲存評估結果

In [ ]:
RESULTS_DIR = DRIVE_PROJECT / "results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# --- Classification report → CSV ---
report_dict = classification_report(
    all_labels, all_preds,
    target_names=["non-clickbait", "clickbait"],
    output_dict=True,
)
report_df = pd.DataFrame(report_dict).transpose()
report_path = RESULTS_DIR / f"{RUN_TAG}_transformer_metrics.csv"
report_df.to_csv(report_path, encoding="utf-8-sig")
print("Metrics saved to:", report_path)
print(report_df.round(4))

# --- Metrics JSON (accuracy + macro_f1) ---
metrics_json = {
    "run_tag": RUN_TAG,
    "accuracy": report_dict["accuracy"],
    "macro_f1": report_dict["macro avg"]["f1-score"],
}
with open(METRICS_OUT, "w") as f:
    json.dump(metrics_json, f, indent=2)
print("JSON metrics saved to:", METRICS_OUT)

# --- Confusion matrix → PNG ---
cm = confusion_matrix(all_labels, all_preds)
fig, ax = plt.subplots(figsize=(5, 4))
im = ax.imshow(cm, cmap="Blues")
plt.colorbar(im, ax=ax)
ax.set_xticks([0, 1]); ax.set_xticklabels(["non-clickbait", "clickbait"])
ax.set_yticks([0, 1]); ax.set_yticklabels(["non-clickbait", "clickbait"])
ax.set_xlabel("Predicted"); ax.set_ylabel("True")
ax.set_title(f"XLM-RoBERTa Confusion Matrix ({RUN_TAG.upper()})")
for i in range(2):
    for j in range(2):
        ax.text(j, i, cm[i, j], ha="center", va="center",
                color="white" if cm[i, j] > cm.max() / 2 else "black")
fig.tight_layout()
cm_path = RESULTS_DIR / f"{RUN_TAG}_transformer_confusion_matrix.png"
fig.savefig(cm_path, dpi=150)
plt.show()
print("Confusion matrix saved to:", cm_path)